# GTZAN Music Genre Classification - Sprint 1: Data Loading & Feature Extraction

## Objective
Load the GTZAN music dataset from `Data_Music/genres_original/` into PostgreSQL, 
extract audio features using Librosa, and populate the database tables 
(`music_genre`, `audio_track`, `features_30_sec`, `features_3_sec`).

## Goal
By the end of this notebook:
- ✓ All 1000 .wav files catalogued in the database
- ✓ Audio features (tempo, MFCCs, spectral data) extracted and stored
- ✓ Database ready for EDA and model training in Sprint 2

## Scope
- Load credentials from `.env` file
- Create PostgreSQL tables using DDL schema
- Scan `genres_original/` folder, extract 1000 tracks
- Compute audio features with Librosa for each track
- Insert all data into PostgreSQL
- Validate data integrity

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# PostgreSQL connectivity
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Audio processing
import librosa

import warnings
warnings.filterwarnings('ignore')

# Suppress audioread macOS warnings
import logging
logging.getLogger('audioread').setLevel(logging.ERROR)

print('✓ Libraries loaded')

✓ Libraries loaded


In [2]:
# ── Load credentials from .env ─────────────────────────────────────────────
load_dotenv()

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_NAME     = 'music_genre_db'

assert DB_USER,     'DB_USER not found in .env'
assert DB_PASSWORD, 'DB_PASSWORD not found in .env'

print(f'✓ Credentials loaded: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

✓ Credentials loaded: ingxrodriguez@localhost:5432/music_genre_db


In [3]:
# ── Create SQLAlchemy engine and test connection ────────────────────────
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Test the connection
try:
    with engine.connect() as conn:
        result = conn.execute(text('SELECT current_database(), current_user;'))
        db_info = result.fetchone()
        print(f'✓ Connected to: {db_info[0]} as {db_info[1]}')
except Exception as e:
    print(f'✗ Connection failed: {e}')

✓ Connected to: music_genre_db as ingxrodriguez


In [4]:
# Define the path to the GTZAN dataset 
GTZAN_PATH = Path('./Data_Music/genres_original')

# Check if the path exist
if GTZAN_PATH.exists():
    print(f'✓ Dataset found at: {GTZAN_PATH.absolute()}')
else:
    print(f'✗ Dataset NOT found at: {GTZAN_PATH.absolute()}')
    print(f'Check that the folder exists in your project')

# List the genres
genres = sorted([d.name for d in GTZAN_PATH.iterdir() if d.is_dir()])
print(f'\nGenres found: {len(genres)}')
for genre in genres:
    print(f'  - {genre}')

✓ Dataset found at: /Users/ingxrodriguez/music-genre-classification/Data_Music/genres_original

Genres found: 10
  - blues
  - classical
  - country
  - disco
  - hiphop
  - jazz
  - metal
  - pop
  - reggae
  - rock


# Create Tables


In [ ]:
# Create music_genre table 
CREATE_GENRE_TABLE = """
DROP TABLE IF EXISTS music_genre CASCADE;

CREATE TABLE music_genre (
  id SERIAL PRIMARY KEY,
  name VARCHAR(50) NOT NULL UNIQUE
);
"""

# Execute
with engine.begin() as conn:
    conn.execute(text(CREATE_GENRE_TABLE))
    print('✓ Table music_genre created')

✓ Table music_genre created


In [ ]:
# Create audio_track table 
CREATE_TRACK_TABLE = """
CREATE TABLE audio_track (
  id SERIAL PRIMARY KEY,
  genre_id INT NOT NULL REFERENCES music_genre(id) ON DELETE CASCADE,
  filename VARCHAR(255) NOT NULL,
  file_path VARCHAR(512)
);

CREATE INDEX idx_audio_track_genre_id ON audio_track(genre_id);
"""

# Execute
with engine.begin() as conn:
    conn.execute(text(CREATE_TRACK_TABLE))
    print('✓ Table audio_track created')

✓ Table audio_track created


In [7]:
# ── Create features_30_sec table ────────────────────────────────────────
CREATE_FEATURES_30_TABLE = """
CREATE TABLE features_30_sec (
  id SERIAL PRIMARY KEY,
  track_id INT NOT NULL REFERENCES audio_track(id) ON DELETE CASCADE,
  length INT,
  tempo DOUBLE PRECISION,
  chroma_mean DOUBLE PRECISION,
  mfcc1_mean DOUBLE PRECISION,
  mfcc2_mean DOUBLE PRECISION,
  mfcc3_mean DOUBLE PRECISION,
  mfcc4_mean DOUBLE PRECISION,
  mfcc5_mean DOUBLE PRECISION,
  mfcc6_mean DOUBLE PRECISION,
  mfcc7_mean DOUBLE PRECISION,
  mfcc8_mean DOUBLE PRECISION,
  mfcc9_mean DOUBLE PRECISION,
  mfcc10_mean DOUBLE PRECISION,
  mfcc11_mean DOUBLE PRECISION,
  mfcc12_mean DOUBLE PRECISION,
  mfcc13_mean DOUBLE PRECISION,
  spectral_centroid DOUBLE PRECISION,
  spectral_rolloff DOUBLE PRECISION,
  zero_crossing_rate DOUBLE PRECISION
);

CREATE UNIQUE INDEX idx_features_30_sec_track_id ON features_30_sec(track_id);
"""

# Execute
with engine.begin() as conn:
    conn.execute(text(CREATE_FEATURES_30_TABLE))
    print('✓ Table features_30_sec created')

✓ Table features_30_sec created


In [ ]:
#  Create features_3_sec table 
CREATE_FEATURES_3_TABLE = """
CREATE TABLE features_3_sec (
  id SERIAL PRIMARY KEY,
  track_id INT NOT NULL REFERENCES audio_track(id) ON DELETE CASCADE,
  segment_num INT NOT NULL,
  start_time_sec DOUBLE PRECISION,
  tempo DOUBLE PRECISION,
  chroma_mean DOUBLE PRECISION,
  mfcc1_mean DOUBLE PRECISION,
  mfcc2_mean DOUBLE PRECISION,
  mfcc3_mean DOUBLE PRECISION,
  mfcc4_mean DOUBLE PRECISION,
  mfcc5_mean DOUBLE PRECISION,
  mfcc6_mean DOUBLE PRECISION,
  mfcc7_mean DOUBLE PRECISION,
  mfcc8_mean DOUBLE PRECISION,
  mfcc9_mean DOUBLE PRECISION,
  mfcc10_mean DOUBLE PRECISION,
  mfcc11_mean DOUBLE PRECISION,
  mfcc12_mean DOUBLE PRECISION,
  mfcc13_mean DOUBLE PRECISION,
  spectral_centroid DOUBLE PRECISION,
  spectral_rolloff DOUBLE PRECISION,
  zero_crossing_rate DOUBLE PRECISION
);

CREATE INDEX idx_features_3_sec_track_id ON features_3_sec(track_id);
CREATE INDEX idx_features_3_sec_segment_num ON features_3_sec(segment_num);
"""

# Execute
with engine.begin() as conn:
    conn.execute(text(CREATE_FEATURES_3_TABLE))
    print('✓ Table features_3_sec created')

✓ Table features_3_sec created


# Insert Genres

In [9]:
# Insert the 10 genres into music_genre
INSERT_GENRES = """
INSERT INTO music_genre (name) VALUES
('blues'),
('classical'),
('country'),
('disco'),
('hiphop'),
('jazz'),
('metal'),
('pop'),
('reggae'),
('rock')
ON CONFLICT (name) DO NOTHING;
"""

# Execute
with engine.begin() as conn:
    conn.execute(text(INSERT_GENRES))
    
# Verify
genres_in_db = pd.read_sql('SELECT * FROM music_genre ORDER BY name', engine)
print(f'✓ {len(genres_in_db)} genres inserted')
print(genres_in_db)

✓ 10 genres inserted
   id       name
0   1      blues
1   2  classical
2   3    country
3   4      disco
4   5     hiphop
5   6       jazz
6   7      metal
7   8        pop
8   9     reggae
9  10       rock


# Extract features and load into the dataset

In [10]:
# Define function to extract audio features 
def extract_features_30sec(audio_path, sr=22050):
    """Extract features from a 30-second audio file"""
    try:
        y, sr = librosa.load(audio_path, sr=sr)
        
        # Tempo
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        
        # Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma)
        
        # MFCC (13 coefficients)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_means = np.mean(mfcc, axis=1)
        
        # Spectral features
        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        
        return {
            'length': len(y),
            'tempo': tempo,
            'chroma_mean': chroma_mean,
            'mfcc1_mean': mfcc_means[0],
            'mfcc2_mean': mfcc_means[1],
            'mfcc3_mean': mfcc_means[2],
            'mfcc4_mean': mfcc_means[3],
            'mfcc5_mean': mfcc_means[4],
            'mfcc6_mean': mfcc_means[5],
            'mfcc7_mean': mfcc_means[6],
            'mfcc8_mean': mfcc_means[7],
            'mfcc9_mean': mfcc_means[8],
            'mfcc10_mean': mfcc_means[9],
            'mfcc11_mean': mfcc_means[10],
            'mfcc12_mean': mfcc_means[11],
            'mfcc13_mean': mfcc_means[12],
            'spectral_centroid': spectral_centroid,
            'spectral_rolloff': spectral_rolloff,
            'zero_crossing_rate': zcr
        }
    except Exception as e:
        print(f'  ✗ Error processing {audio_path.name}: {e}')
        return None

print('✓ Feature extraction function defined')

✓ Feature extraction function defined


In [11]:
# Process all .wav files and extract features 
all_features = []
total_files = sum(len(list(GTZAN_PATH.glob(f'{genre}/*.wav'))) 
                  for genre in genres)

print(f'Processing {total_files} audio files...\n')

for genre in genres:
    genre_path = GTZAN_PATH / genre
    wav_files = sorted(genre_path.glob('*.wav'))
    
    print(f'{genre:12} → {len(wav_files):3} files', end=' ')
    
    for wav_file in wav_files:
        features = extract_features_30sec(wav_file)
        if features:
            features['genre'] = genre
            features['filename'] = wav_file.name
            features['file_path'] = str(wav_file.absolute())
            all_features.append(features)
    
    print(f'✓ ({len([f for f in all_features if f["genre"] == genre])} extracted)')

print(f'\n✓ Total features extracted: {len(all_features)}')

Processing 1000 audio files...

blues        → 100 files ✓ (100 extracted)
classical    → 100 files ✓ (100 extracted)
country      → 100 files ✓ (100 extracted)
disco        → 100 files ✓ (100 extracted)
hiphop       → 100 files ✓ (100 extracted)
jazz         → 100 files 

Exception ignored in: <function CFObject.__del__ at 0x1668b5940>
Traceback (most recent call last):
  File "/Users/ingxrodriguez/miniconda3/envs/power_snake/lib/python3.11/site-packages/audioread/macca.py", line 135, in __del__
    _corefoundation.CFRelease(self._obj)
                              ^^^^^^^^^
AttributeError: 'CFURL' object has no attribute '_obj'
Exception ignored in: <function ExtAudioFile.__del__ at 0x1668b6340>
Traceback (most recent call last):
  File "/Users/ingxrodriguez/miniconda3/envs/power_snake/lib/python3.11/site-packages/audioread/macca.py", line 336, in __del__
    self.close()
  File "/Users/ingxrodriguez/miniconda3/envs/power_snake/lib/python3.11/site-packages/audioread/macca.py", line 330, in close
    if not self.closed:
           ^^^^^^^^^^^
AttributeError: 'ExtAudioFile' object has no attribute 'closed'


  ✗ Error processing jazz.00054.wav: 'PosixPath' object has no attribute 'encode'
✓ (99 extracted)
metal        → 100 files ✓ (100 extracted)
pop          → 100 files ✓ (100 extracted)
reggae       → 100 files ✓ (100 extracted)
rock         → 100 files ✓ (100 extracted)

✓ Total features extracted: 999


# Convert to DataFrame and insert into the database



In [12]:
# Convert to DataFrame 
df_features = pd.DataFrame(all_features)
print(f'DataFrame shape: {df_features.shape}')
print(f'\nFirst row:')
print(df_features.head(1))

# Get genre IDs
genre_map = pd.read_sql('SELECT id, name FROM music_genre', engine).set_index('name')['id'].to_dict()

# Add track_id column (will be auto-generated in DB, but we need it for relationships)
df_tracks = df_features[['genre', 'filename', 'file_path']].copy()
df_tracks['genre_id'] = df_tracks['genre'].map(genre_map)
df_tracks = df_tracks[['genre_id', 'filename', 'file_path']]

# Load tracks to database
df_tracks.to_sql('audio_track', engine, if_exists='append', index=False)
print(f'\n✓ Inserted {len(df_tracks)} tracks into audio_track')

DataFrame shape: (999, 22)

First row:
   length         tempo  chroma_mean  mfcc1_mean  mfcc2_mean  mfcc3_mean  \
0  661794  [123.046875]     0.350129 -113.598824  121.570671  -19.162262   

   mfcc4_mean  mfcc5_mean  mfcc6_mean  mfcc7_mean  ...  mfcc10_mean  \
0   42.363941   -6.362266   18.621931  -13.699734  ...    10.970945   

   mfcc11_mean  mfcc12_mean  mfcc13_mean  spectral_centroid  spectral_rolloff  \
0    -8.326061     8.802088    -3.669941        1784.122641        3805.72303   

   zero_crossing_rate  genre         filename  \
0            0.083045  blues  blues.00000.wav   

                                           file_path  
0  /Users/ingxrodriguez/music-genre-classificatio...  

[1 rows x 22 columns]

✓ Inserted 999 tracks into audio_track


In [ ]:
# Clean features_30_sec table 
with engine.begin() as conn:
    conn.execute(text('DELETE FROM features_30_sec'))
    print('✓ features_30_sec cleaned')

✓ features_30_sec cleaned


In [22]:
# Fix tempo values (convert from numpy array to float) 
df_features['tempo'] = df_features['tempo'].apply(
    lambda x: float(x[0]) if isinstance(x, np.ndarray) else float(x)
)

print('✓ Tempo values converted to float')
print(f'Sample tempos: {df_features["tempo"].head()}')

✓ Tempo values converted to float
Sample tempos: 0    123.046875
1     67.999589
2    161.499023
3     63.024009
4    135.999178
Name: tempo, dtype: float64


In [23]:
# Get track IDs that were just created 
df_tracks_db = pd.read_sql('SELECT id, filename FROM audio_track', engine)
filename_to_id = dict(zip(df_tracks_db['filename'], df_tracks_db['id']))

# Prepare features_30_sec data
df_features_30 = df_features[[
    'filename', 'length', 'tempo', 'chroma_mean',
    'mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean',
    'mfcc6_mean', 'mfcc7_mean', 'mfcc8_mean', 'mfcc9_mean', 'mfcc10_mean',
    'mfcc11_mean', 'mfcc12_mean', 'mfcc13_mean',
    'spectral_centroid', 'spectral_rolloff', 'zero_crossing_rate'
]].copy()

df_features_30['track_id'] = df_features_30['filename'].map(filename_to_id)
df_features_30 = df_features_30.drop('filename', axis=1)

# Reorder columns
cols = ['track_id'] + [c for c in df_features_30.columns if c != 'track_id']
df_features_30 = df_features_30[cols]

# Load to database
df_features_30.to_sql('features_30_sec', engine, if_exists='append', index=False)
print(f'✓ Inserted {len(df_features_30)} feature records into features_30_sec')

✓ Inserted 999 feature records into features_30_sec


In [24]:
# ── Verify data loaded correctly ────────────────────────────────────────

# Count genres
genre_count = pd.read_sql('SELECT COUNT(*) as count FROM music_genre', engine)
print(f'Genres: {genre_count["count"].values[0]}')

# Count tracks
track_count = pd.read_sql('SELECT COUNT(*) as count FROM audio_track', engine)
print(f'Audio tracks: {track_count["count"].values[0]}')

# Count features
feature_count = pd.read_sql('SELECT COUNT(*) as count FROM features_30_sec', engine)
print(f'Features_30_sec: {feature_count["count"].values[0]}')

# Check 1:1 relationship
print('\n── Verify 1:1 relationship ──')
print(f'Tracks:   {track_count["count"].values[0]}')
print(f'Features: {feature_count["count"].values[0]}')
if track_count["count"].values[0] == feature_count["count"].values[0]:
    print('✓ 1:1 relationship OK')
else:
    print('✗ Mismatch!')

# Show sample data
print('\n── Sample data ──')
sample = pd.read_sql('''
    SELECT at.filename, mg.name, f.tempo, f.mfcc1_mean, f.spectral_centroid
    FROM audio_track at
    JOIN music_genre mg ON at.genre_id = mg.id
    JOIN features_30_sec f ON at.id = f.track_id
    LIMIT 5
''', engine)
display(sample)

Genres: 10
Audio tracks: 999
Features_30_sec: 999

── Verify 1:1 relationship ──
Tracks:   999
Features: 999
✓ 1:1 relationship OK

── Sample data ──


,filename,name,tempo,mfcc1_mean,spectral_centroid
0,blues.00000.wav,blues,123.046875,-113.598824,1784.122641
1,blues.00001.wav,blues,67.999589,-207.523834,1530.261767
2,blues.00002.wav,blues,161.499023,-90.757164,1552.832481
3,blues.00003.wav,blues,63.024009,-199.575134,1070.153418
4,blues.00004.wav,blues,135.999178,-160.354172,1835.128513


In [25]:
# Extract features for 3-second segments 
# Each 30-second track is divided into 10 segments of 3 seconds
# So we'll have 999 tracks × 10 segments = 9990 feature records

features_3_sec_list = []

print('Extracting 3-second segment features...\n')

for genre in genres:
    genre_path = GTZAN_PATH / genre
    wav_files = sorted(genre_path.glob('*.wav'))
    
    for wav_file in wav_files:
        try:
            y, sr = librosa.load(wav_file, sr=22050)
            
            # Divide into 10 segments of ~3 seconds
            segment_length = len(y) // 10
            
            for segment_num in range(10):
                start_sample = segment_num * segment_length
                end_sample = (segment_num + 1) * segment_length
                y_segment = y[start_sample:end_sample]
                
                # Extract features for this segment
                tempo, _ = librosa.beat.beat_track(y=y_segment, sr=sr)
                chroma = librosa.feature.chroma_stft(y=y_segment, sr=sr)
                chroma_mean = np.mean(chroma)
                mfcc = librosa.feature.mfcc(y=y_segment, sr=sr, n_mfcc=13)
                mfcc_means = np.mean(mfcc, axis=1)
                spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y_segment, sr=sr))
                spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=y_segment, sr=sr))
                zcr = np.mean(librosa.feature.zero_crossing_rate(y_segment))
                
                record = {
                    'filename': wav_file.name,
                    'segment_num': segment_num,
                    'start_time_sec': (segment_num * 3),
                    'tempo': float(tempo[0]) if isinstance(tempo, np.ndarray) else float(tempo),
                    'chroma_mean': chroma_mean,
                    'mfcc1_mean': mfcc_means[0],
                    'mfcc2_mean': mfcc_means[1],
                    'mfcc3_mean': mfcc_means[2],
                    'mfcc4_mean': mfcc_means[3],
                    'mfcc5_mean': mfcc_means[4],
                    'mfcc6_mean': mfcc_means[5],
                    'mfcc7_mean': mfcc_means[6],
                    'mfcc8_mean': mfcc_means[7],
                    'mfcc9_mean': mfcc_means[8],
                    'mfcc10_mean': mfcc_means[9],
                    'mfcc11_mean': mfcc_means[10],
                    'mfcc12_mean': mfcc_means[11],
                    'mfcc13_mean': mfcc_means[12],
                    'spectral_centroid': spectral_centroid,
                    'spectral_rolloff': spectral_rolloff,
                    'zero_crossing_rate': zcr
                }
                features_3_sec_list.append(record)
        except Exception as e:
            pass

print(f'✓ Total 3-sec features extracted: {len(features_3_sec_list)}')

# Convert to DataFrame
df_features_3 = pd.DataFrame(features_3_sec_list)
print(f'DataFrame shape: {df_features_3.shape}')

Extracting 3-second segment features...



Exception ignored in: <function CFObject.__del__ at 0x1668b5940>
Traceback (most recent call last):
  File "/Users/ingxrodriguez/miniconda3/envs/power_snake/lib/python3.11/site-packages/audioread/macca.py", line 135, in __del__
    _corefoundation.CFRelease(self._obj)
                              ^^^^^^^^^
AttributeError: 'CFURL' object has no attribute '_obj'
Exception ignored in: <function ExtAudioFile.__del__ at 0x1668b6340>
Traceback (most recent call last):
  File "/Users/ingxrodriguez/miniconda3/envs/power_snake/lib/python3.11/site-packages/audioread/macca.py", line 336, in __del__
    self.close()
  File "/Users/ingxrodriguez/miniconda3/envs/power_snake/lib/python3.11/site-packages/audioread/macca.py", line 330, in close
    if not self.closed:
           ^^^^^^^^^^^
AttributeError: 'ExtAudioFile' object has no attribute 'closed'


✓ Total 3-sec features extracted: 9990
DataFrame shape: (9990, 21)


In [26]:
# Insert features_3_sec into database 

# Get filename to track_id mapping
df_tracks_db = pd.read_sql('SELECT id, filename FROM audio_track', engine)
filename_to_id = dict(zip(df_tracks_db['filename'], df_tracks_db['id']))

# Add track_id to the DataFrame
df_features_3['track_id'] = df_features_3['filename'].map(filename_to_id)
df_features_3 = df_features_3.drop('filename', axis=1)

# Reorder columns
cols = ['track_id', 'segment_num', 'start_time_sec'] + [c for c in df_features_3.columns 
        if c not in ['track_id', 'segment_num', 'start_time_sec']]
df_features_3 = df_features_3[cols]

# Load to database
df_features_3.to_sql('features_3_sec', engine, if_exists='append', index=False)
print(f'✓ Inserted {len(df_features_3)} records into features_3_sec')

✓ Inserted 9990 records into features_3_sec


In [28]:
# ── Final Check    

print('=== FINAL DATABASE STATUS ===\n')

# Count everything
genres = pd.read_sql('SELECT COUNT(*) as count FROM music_genre', engine)['count'].values[0]
tracks = pd.read_sql('SELECT COUNT(*) as count FROM audio_track', engine)['count'].values[0]
features_30 = pd.read_sql('SELECT COUNT(*) as count FROM features_30_sec', engine)['count'].values[0]
features_3 = pd.read_sql('SELECT COUNT(*) as count FROM features_3_sec', engine)['count'].values[0]

print(f'Genres:           {genres:,}')
print(f'Audio tracks:     {tracks:,}')
print(f'Features_30_sec:  {features_30:,}')
print(f'Features_3_sec:   {features_3:,}')

print(f'\n=== RELATIONSHIPS ===')
print(f'Expected features_3_sec: {tracks} × 10 = {tracks * 10:,}')
print(f'Actual features_3_sec:   {features_3:,}')
if features_3 == tracks * 10:
    print('✓ 1:M relationship OK')
else:
    print(f'⚠ Difference: {features_3 - (tracks * 10)}')

print(f'\n=== SAMPLE DATA ===')
sample = pd.read_sql('''
    SELECT 
        mg.name as genre,
        COUNT(DISTINCT at.id) as track_count,
        COUNT(DISTINCT f3.id) as segment_count
    FROM music_genre mg
    LEFT JOIN audio_track at ON mg.id = at.genre_id
    LEFT JOIN features_3_sec f3 ON at.id = f3.track_id
    GROUP BY mg.name
    ORDER BY mg.name
''', engine)
display(sample)

print('\n✓ Database is ready for EDA , Pre-Processing and modeling!')

=== FINAL DATABASE STATUS ===

Genres:           10
Audio tracks:     999
Features_30_sec:  999
Features_3_sec:   9,990

=== RELATIONSHIPS ===
Expected features_3_sec: 999 × 10 = 9,990
Actual features_3_sec:   9,990
✓ 1:M relationship OK

=== SAMPLE DATA ===


,genre,track_count,segment_count
0,blues,100,1000
1,classical,100,1000
2,country,100,1000
3,disco,100,1000
4,hiphop,100,1000
5,jazz,99,990
6,metal,100,1000
7,pop,100,1000
8,reggae,100,1000
9,rock,100,1000



✓ Database is ready for EDA , Pre-Processing and modeling!
